# fase_5 - script_afrida Migration

This notebook handles migration of database from old DB to new DB for fase 5.

**Purpose**: Benerin database lama ke database baru untuk bagian [NAMA TABEL]

In [1]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')

db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)
print(f'Connected to future database: {config["db_future"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_juni
Connected to new database: dataleap_v5_migration
Connected to future database: db_future


## 2. Ambil Data dari DB Lama

In [3]:
import pandas as pd
import pickle

# =========================================================
# 1. LOAD MAPPING DARI FILE
# =========================================================
print("="*70)
print("📂 Memuat mapping untuk presensi_siswa")
print("="*70)

# 1a. Load mapping dari fase_4_afrida.pkl
print("\n📂 Memuat fase_4_afrida.pkl...")
with open('fase_4_afrida.pkl', 'rb') as f:
    data_fase4 = pickle.load(f)

mapping_id_jadwal_detail = data_fase4['mapping_id_jadwal_detail']
print(f"✅ Mapping id_jadwal_detail: {len(mapping_id_jadwal_detail)} entri")

# 1b. Load mapping siswa dari mapping_siswa.pkl
print("\n📂 Memuat mapping_siswa.pkl...")
with open('mapping_siswa.pkl', 'rb') as f:
    mapping_siswa_df = pickle.load(f)

print(f"Type mapping_siswa: {type(mapping_siswa_df)}")
if isinstance(mapping_siswa_df, pd.DataFrame):
    print(f"Shape: {mapping_siswa_df.shape}")
    print(f"Kolom: {mapping_siswa_df.columns.tolist()}")
    
    # Konversi DataFrame -> Dictionary
    kolom_old = mapping_siswa_df.columns[0]  # asumsi kolom pertama = old_id
    kolom_new = mapping_siswa_df.columns[1]  # asumsi kolom kedua = new_id
    
    mapping_id_siswa = dict(zip(
        mapping_siswa_df[kolom_old].astype(str),
        mapping_siswa_df[kolom_new]
    ))
    print(f"✅ Mapping siswa dikonversi: {len(mapping_id_siswa)} entri")
else:
    mapping_id_siswa = mapping_siswa_df
    print(f"✅ Mapping siswa siap: {len(mapping_id_siswa)} entri")

# =========================================================
# 2. PROSES DATA presensi_siswa
# =========================================================
print("\n" + "="*70)
print("⚡ Proses presensi_siswa")
print("="*70)

# Ambil data dari DB lama
query = "SELECT idpresensi_siswa, idjadwaldetil, idsiswa, waktu, status FROM presensi_siswa"
cursor_old.execute(query)
rows = cursor_old.fetchall()

# Buat DataFrame dengan nama kolom yang jelas
df = pd.DataFrame(rows, columns=['idpresensi_siswa', 'idjadwaldetil', 'idsiswa', 'waktu', 'status'])
print(f"Data mentah: {len(df)} baris")

# Rename kolom sesuai target
rename_dict = {
    'idpresensi_siswa': 'id_presensi_siswa_lama',  # akan di-drop
    'idjadwaldetil': 'id_jadwal_detail_old',
    'idsiswa': 'id_siswa_old',
    'waktu': 'waktu_presensi',
    'status': 'status_presensi'
}
df.rename(columns=rename_dict, inplace=True)

# =========================================================
# 3. MAPPING id_jadwal_detail
# =========================================================
print("\n📌 Mapping id_jadwal_detail...")
df['id_jadwal_detail'] = df['id_jadwal_detail_old'].astype(str).map(mapping_id_jadwal_detail)

before = len(df)
df.dropna(subset=['id_jadwal_detail'], inplace=True)
print(f"  Baris orphan detail dibuang: {before - len(df)}")
df['id_jadwal_detail'] = df['id_jadwal_detail'].astype('int64')

# =========================================================
# 4. MAPPING id_siswa
# =========================================================
print("\n📌 Mapping id_siswa...")
df['id_siswa'] = df['id_siswa_old'].astype(str).map(mapping_id_siswa)

before_siswa = len(df)
df.dropna(subset=['id_siswa'], inplace=True)
print(f"  Baris orphan siswa dibuang: {before_siswa - len(df)}")
df['id_siswa'] = df['id_siswa'].astype('int64')

# =========================================================
# 5. KONVERSI TIPE DATA & FINALISASI
# =========================================================
print("\n📌 Konversi tipe data...")
df['waktu_presensi'] = pd.to_datetime(df['waktu_presensi'])
df['status_presensi'] = df['status_presensi'].astype('int8')

# Hapus kolom sementara
df.drop(columns=['id_presensi_siswa_lama', 'id_jadwal_detail_old', 'id_siswa_old'], inplace=True)

# Tambahkan id internal (auto increment)
df.insert(0, 'id', range(1, len(df) + 1))

# =========================================================
# 6. HASIL AKHIR
# =========================================================
df_presensi_siswa = df.copy()

print(f"\n✅ df_presensi_siswa siap. Shape: {df_presensi_siswa.shape}")
print(f"   Kolom: {df_presensi_siswa.columns.tolist()}")

# =========================================================
# 7. VALIDASI FK
# =========================================================
print("\n" + "="*70)
print("🔍 VALIDASI FK presensi_siswa")
print("="*70)

# Validasi ke jadwal_detail
fk_detail_values = set(df_presensi_siswa['id_jadwal_detail'])
valid_detail_keys = set(mapping_id_jadwal_detail.values())
invalid_detail = fk_detail_values - valid_detail_keys
print(f"\n  FK ke jadwal_detail:")
print(f"    Total nilai unik: {len(fk_detail_values)}")
print(f"    Invalid: {len(invalid_detail)}")
if len(invalid_detail) == 0:
    print("    ✅ SEMUA VALID!")
else:
    print(f"    ❌ Invalid: {list(invalid_detail)[:5]}")

# Validasi ke siswa
fk_siswa_values = set(df_presensi_siswa['id_siswa'])
valid_siswa_keys = set(mapping_id_siswa.values())
invalid_siswa = fk_siswa_values - valid_siswa_keys
print(f"\n  FK ke siswa:")
print(f"    Total nilai unik: {len(fk_siswa_values)}")
print(f"    Invalid: {len(invalid_siswa)}")
if len(invalid_siswa) == 0:
    print("    ✅ SEMUA VALID!")
else:
    print(f"    ❌ Invalid: {list(invalid_siswa)[:5]}")

# =========================================================
# 8. SIMPAN KE fase_5_afrida.pkl
# =========================================================
print("\n" + "="*70)
print("💾 Menyimpan ke fase_5_afrida.pkl")
print("="*70)

# Load file fase_5_afrida.pkl jika ada, atau buat baru
try:
    with open('fase_5_afrida.pkl', 'rb') as f:
        data_fase5 = pickle.load(f)
    print("✅ fase_5_afrida.pkl ditemukan, melanjutkan...")
except FileNotFoundError:
    data_fase5 = {}
    print("📂 fase_5_afrida.pkl belum ada, membuat baru...")

# Tambahkan data presensi_siswa
data_fase5['presensi_siswa'] = df_presensi_siswa
data_fase5['mapping_id_presensi_siswa'] = dict(zip(
    df_presensi_siswa['id'].astype(str),
    df_presensi_siswa['id']
))

# Simpan ke file
with open('fase_5_afrida.pkl', 'wb') as f:
    pickle.dump(data_fase5, f)

print(f"✅ presensi_siswa disimpan ke fase_5_afrida.pkl")
print(f"   Shape: {df_presensi_siswa.shape}")
print(f"   Total data: {len(df_presensi_siswa)} baris")

print("\n✅ Proses presensi_siswa selesai!")

📂 Memuat mapping untuk presensi_siswa

📂 Memuat fase_4_afrida.pkl...


✅ Mapping id_jadwal_detail: 17312 entri

📂 Memuat mapping_siswa.pkl...
Type mapping_siswa: <class 'pandas.core.frame.DataFrame'>
Shape: (1500, 2)
Kolom: ['idsiswa_lama', 'id_siswa_baru']
✅ Mapping siswa dikonversi: 1500 entri

⚡ Proses presensi_siswa
Data mentah: 108795 baris

📌 Mapping id_jadwal_detail...
  Baris orphan detail dibuang: 0

📌 Mapping id_siswa...
  Baris orphan siswa dibuang: 0

📌 Konversi tipe data...

✅ df_presensi_siswa siap. Shape: (108795, 5)
   Kolom: ['id', 'waktu_presensi', 'status_presensi', 'id_jadwal_detail', 'id_siswa']

🔍 VALIDASI FK presensi_siswa

  FK ke jadwal_detail:
    Total nilai unik: 14200
    Invalid: 0
    ✅ SEMUA VALID!

  FK ke siswa:
    Total nilai unik: 1319
    Invalid: 0
    ✅ SEMUA VALID!

💾 Menyimpan ke fase_5_afrida.pkl
✅ fase_5_afrida.pkl ditemukan, melanjutkan...
✅ presensi_siswa disimpan ke fase_5_afrida.pkl
   Shape: (108795, 5)
   Total data: 108795 baris

✅ Proses presensi_siswa selesai!


In [4]:
# Verifikasi isi fase_5_afrida.pkl
with open('fase_5_afrida.pkl', 'rb') as f:
    data = pickle.load(f)

print("\n📋 Isi fase_5_afrida.pkl:")
print(f"  Key: {list(data.keys())}")
print(f"  presensi_siswa shape: {data['presensi_siswa'].shape}")
print(f"  mapping_id_presensi_siswa: {len(data['mapping_id_presensi_siswa'])} entri")

print("\nSample presensi_siswa:")
print(data['presensi_siswa'][['id', 'id_jadwal_detail', 'id_siswa', 'waktu_presensi']].head())


📋 Isi fase_5_afrida.pkl:
  Key: ['presensi_siswa', 'catatan_siswa', 'followup_cs', 'mapping_id_presensi_siswa']
  presensi_siswa shape: (108795, 5)
  mapping_id_presensi_siswa: 108795 entri

Sample presensi_siswa:
   id  id_jadwal_detail  id_siswa      waktu_presensi
0   1               721       347 2023-07-05 16:24:39
1   2               721       348 2023-07-05 16:24:40
2   3               331       363 2023-07-05 16:41:58
3   4               331       380 2023-07-05 16:42:00
4   5               331       381 2023-07-05 16:42:00


In [5]:
display(df_presensi_siswa.head())

,id,waktu_presensi,status_presensi,id_jadwal_detail,id_siswa
0,1,2023-07-05 16:24:39,1,721,347
1,2,2023-07-05 16:24:40,1,721,348
2,3,2023-07-05 16:41:58,1,331,363
3,4,2023-07-05 16:42:00,1,331,380
4,5,2023-07-05 16:42:00,1,331,381


In [6]:
import pandas as pd
import pickle

# =========================================================
# 1. LOAD SEMUA MAPPING YANG DIBUTUHKAN
# =========================================================
print("="*70)
print("📂 Memuat mapping untuk catatan_siswa")
print("="*70)

# 1a. Load mapping dari fase_4_afrida.pkl
print("\n📂 Memuat fase_4_afrida.pkl...")
with open('fase_4_afrida.pkl', 'rb') as f:
    data_fase4 = pickle.load(f)

mapping_id_jadwal = data_fase4['mapping_id_jadwal']
mapping_id_jadwal_detail = data_fase4['mapping_id_jadwal_detail']
print(f"✅ Mapping id_jadwal: {len(mapping_id_jadwal)} entri")
print(f"✅ Mapping id_jadwal_detail: {len(mapping_id_jadwal_detail)} entri")

# 1b. Load mapping siswa dari mapping_siswa.pkl
print("\n📂 Memuat mapping_siswa.pkl...")
with open('mapping_siswa.pkl', 'rb') as f:
    mapping_siswa_df = pickle.load(f)

if isinstance(mapping_siswa_df, pd.DataFrame):
    print(f"Shape: {mapping_siswa_df.shape}")
    print(f"Kolom: {mapping_siswa_df.columns.tolist()}")
    
    # Konversi DataFrame -> Dictionary
    kolom_old = mapping_siswa_df.columns[0]
    kolom_new = mapping_siswa_df.columns[1]
    
    mapping_id_siswa = dict(zip(
        mapping_siswa_df[kolom_old].astype(str),
        mapping_siswa_df[kolom_new]
    ))
    print(f"✅ Mapping siswa dikonversi: {len(mapping_id_siswa)} entri")
else:
    mapping_id_siswa = mapping_siswa_df
    print(f"✅ Mapping siswa siap: {len(mapping_id_siswa)} entri")

# 1c. Ambil mapping idjadwaldetil -> idjadwal dari DB lama
print("\n📂 Mengambil mapping detail -> jadwal dari DB lama...")
query_jd = "SELECT idjadwaldetil, idjadwal FROM jadwal_detil"
cursor_old.execute(query_jd)
rows_jd = cursor_old.fetchall()
mapping_detil_to_jadwal = {str(row['idjadwaldetil']): str(row['idjadwal']) for row in rows_jd}
print(f"✅ Mapping detail -> jadwal: {len(mapping_detil_to_jadwal)} entri")

# =========================================================
# 2. PROSES DATA catatan_siswa
# =========================================================
print("\n" + "="*70)
print("⚡ Proses catatan_siswa")
print("="*70)

# 2a. Ambil data dari DB lama
query_cs = "SELECT idcatatan_siswa, idjadwaldetil, idsiswa, catatan FROM catatan_siswa"
cursor_old.execute(query_cs)
rows_cs = cursor_old.fetchall()
df_cs = pd.DataFrame(rows_cs, columns=['idcatatan_siswa', 'idjadwaldetil', 'idsiswa', 'catatan'])
print(f"Data mentah: {len(df_cs)} baris")

# 2b. Rename kolom
rename_cs = {
    'idcatatan_siswa': 'id_cs_lama',
    'idjadwaldetil': 'id_jadwal_detail_old',
    'idsiswa': 'id_siswa_old',
    'catatan': 'catatan_cs'
}
df_cs.rename(columns=rename_cs, inplace=True)

# =========================================================
# 3. MAPPING id_jadwal_detail
# =========================================================
print("\n📌 Mapping id_jadwal_detail...")
df_cs['id_jadwal_detail'] = df_cs['id_jadwal_detail_old'].astype(str).map(mapping_id_jadwal_detail)

before = len(df_cs)
df_cs.dropna(subset=['id_jadwal_detail'], inplace=True)
print(f"  Baris orphan detail dibuang: {before - len(df_cs)}")
df_cs['id_jadwal_detail'] = df_cs['id_jadwal_detail'].astype('int64')

# =========================================================
# 4. MAPPING id_jadwal (via detail -> jadwal)
# =========================================================
print("\n📌 Mapping id_jadwal (via detail -> jadwal)...")

# 4a. Dapatkan id_jadwal_lama dari id_jadwal_detail_lama
df_cs['id_jadwal_lama'] = df_cs['id_jadwal_detail_old'].map(mapping_detil_to_jadwal)

# 4b. Mapping id_jadwal_lama -> id_jadwal_baru
df_cs['id_jadwal'] = df_cs['id_jadwal_lama'].map(mapping_id_jadwal)

before_jadwal = len(df_cs)
df_cs.dropna(subset=['id_jadwal'], inplace=True)
print(f"  Baris tanpa id_jadwal valid dibuang: {before_jadwal - len(df_cs)}")
df_cs['id_jadwal'] = df_cs['id_jadwal'].astype('int64')

# =========================================================
# 5. MAPPING id_siswa
# =========================================================
print("\n📌 Mapping id_siswa...")
df_cs['id_siswa'] = df_cs['id_siswa_old'].astype(str).map(mapping_id_siswa)

before_siswa = len(df_cs)
df_cs.dropna(subset=['id_siswa'], inplace=True)
print(f"  Baris orphan siswa dibuang: {before_siswa - len(df_cs)}")
df_cs['id_siswa'] = df_cs['id_siswa'].astype('int64')

# =========================================================
# 6. GENERATE ID BARU & MAPPING
# =========================================================
print("\n📌 Generate id_cs baru...")

# Urutkan berdasarkan id_cs_lama untuk konsistensi
df_cs.sort_values('id_cs_lama', inplace=True)
df_cs.reset_index(drop=True, inplace=True)

# Simpan old_ids untuk mapping
old_ids_cs = df_cs['id_cs_lama'].copy()

# Tambahkan id_cs baru (auto increment)
df_cs.insert(0, 'id_cs', range(1, len(df_cs) + 1))

# Buat mapping untuk followup_cs
mapping_id_cs = dict(zip(old_ids_cs.astype(str), df_cs['id_cs']))
print(f"✅ Mapping id_cs: {len(mapping_id_cs)} entri")

# =========================================================
# 7. TAMBAHKAN KOLOM id_karyawan & tanggal (nullable)
# =========================================================
print("\n📌 Menambahkan kolom id_karyawan & tanggal...")
df_cs['id_karyawan'] = None   # FK ke tabel karyawan, nullable
df_cs['tanggal'] = None       # date, nullable
print("✅ Kolom id_karyawan & tanggal ditambahkan (NULL)")

# =========================================================
# 8. FINALISASI
# =========================================================
print("\n📌 Finalisasi...")

# Hapus kolom sementara
kolom_buang = ['id_cs_lama', 'id_jadwal_detail_old', 'id_siswa_old', 'id_jadwal_lama']
df_cs.drop(columns=kolom_buang, inplace=True, errors='ignore')

# Susun ulang kolom agar rapi
kolom_final = ['id_cs', 'id_jadwal', 'id_jadwal_detail', 'id_siswa', 'catatan_cs', 'id_karyawan', 'tanggal']
df_cs = df_cs[kolom_final]

# =========================================================
# 9. HASIL AKHIR
# =========================================================
df_catatan_siswa = df_cs.copy()

print(f"\n✅ df_catatan_siswa siap. Shape: {df_catatan_siswa.shape}")
print(f"   Kolom: {df_catatan_siswa.columns.tolist()}")
print(f"   Tipe id_siswa: {df_catatan_siswa['id_siswa'].dtype}")
print(f"   Tipe id_jadwal: {df_catatan_siswa['id_jadwal'].dtype}")
print(f"   Tipe id_jadwal_detail: {df_catatan_siswa['id_jadwal_detail'].dtype}")

print("\n📋 Sample data catatan_siswa:")
display(df_catatan_siswa.head())

# =========================================================
# 10. VALIDASI FK
# =========================================================
print("\n" + "="*70)
print("🔍 VALIDASI FK catatan_siswa")
print("="*70)

# Validasi ke jadwal
fk_jadwal_values = set(df_catatan_siswa['id_jadwal'])
valid_jadwal_keys = set(mapping_id_jadwal.values())
invalid_jadwal = fk_jadwal_values - valid_jadwal_keys
print(f"\n  FK ke jadwal:")
print(f"    Total nilai unik: {len(fk_jadwal_values)}")
print(f"    Invalid: {len(invalid_jadwal)}")
print("    ✅ SEMUA VALID!" if len(invalid_jadwal) == 0 else f"    ❌ Invalid: {list(invalid_jadwal)[:5]}")

# Validasi ke jadwal_detail
fk_detail_values = set(df_catatan_siswa['id_jadwal_detail'])
valid_detail_keys = set(mapping_id_jadwal_detail.values())
invalid_detail = fk_detail_values - valid_detail_keys
print(f"\n  FK ke jadwal_detail:")
print(f"    Total nilai unik: {len(fk_detail_values)}")
print(f"    Invalid: {len(invalid_detail)}")
print("    ✅ SEMUA VALID!" if len(invalid_detail) == 0 else f"    ❌ Invalid: {list(invalid_detail)[:5]}")

# Validasi ke siswa
fk_siswa_values = set(df_catatan_siswa['id_siswa'])
valid_siswa_keys = set(mapping_id_siswa.values())
invalid_siswa = fk_siswa_values - valid_siswa_keys
print(f"\n  FK ke siswa:")
print(f"    Total nilai unik: {len(fk_siswa_values)}")
print(f"    Invalid: {len(invalid_siswa)}")
print("    ✅ SEMUA VALID!" if len(invalid_siswa) == 0 else f"    ❌ Invalid: {list(invalid_siswa)[:5]}")

# =========================================================
# 11. SIMPAN KE fase_5_afrida.pkl
# =========================================================
print("\n" + "="*70)
print("💾 Menyimpan ke fase_5_afrida.pkl")
print("="*70)

# Load file fase_5_afrida.pkl jika ada
try:
    with open('fase_5_afrida.pkl', 'rb') as f:
        data_fase5 = pickle.load(f)
    print("✅ fase_5_afrida.pkl ditemukan, melanjutkan...")
except FileNotFoundError:
    data_fase5 = {}
    print("📂 fase_5_afrida.pkl belum ada, membuat baru...")

# Tambahkan data catatan_siswa
data_fase5['catatan_siswa'] = df_catatan_siswa
data_fase5['mapping_id_catatan_siswa'] = mapping_id_cs

# Simpan ke file
with open('fase_5_afrida.pkl', 'wb') as f:
    pickle.dump(data_fase5, f)

print(f"✅ catatan_siswa disimpan ke fase_5_afrida.pkl")
print(f"   Shape: {df_catatan_siswa.shape}")
print(f"   Total data: {len(df_catatan_siswa)} baris")
print(f"   Mapping id_cs: {len(mapping_id_cs)} entri")

print("\n✅ Proses catatan_siswa selesai!")

📂 Memuat mapping untuk catatan_siswa

📂 Memuat fase_4_afrida.pkl...
✅ Mapping id_jadwal: 556 entri
✅ Mapping id_jadwal_detail: 17312 entri

📂 Memuat mapping_siswa.pkl...
Shape: (1500, 2)
Kolom: ['idsiswa_lama', 'id_siswa_baru']
✅ Mapping siswa dikonversi: 1500 entri

📂 Mengambil mapping detail -> jadwal dari DB lama...
✅ Mapping detail -> jadwal: 17322 entri

⚡ Proses catatan_siswa
Data mentah: 1535 baris

📌 Mapping id_jadwal_detail...
  Baris orphan detail dibuang: 0

📌 Mapping id_jadwal (via detail -> jadwal)...
  Baris tanpa id_jadwal valid dibuang: 0

📌 Mapping id_siswa...
  Baris orphan siswa dibuang: 0

📌 Generate id_cs baru...
✅ Mapping id_cs: 1535 entri

📌 Menambahkan kolom id_karyawan & tanggal...
✅ Kolom id_karyawan & tanggal ditambahkan (NULL)

📌 Finalisasi...

✅ df_catatan_siswa siap. Shape: (1535, 7)
   Kolom: ['id_cs', 'id_jadwal', 'id_jadwal_detail', 'id_siswa', 'catatan_cs', 'id_karyawan', 'tanggal']
   Tipe id_siswa: int64
   Tipe id_jadwal: int64
   Tipe id_jadwal_det

,id_cs,id_jadwal,id_jadwal_detail,id_siswa,catatan_cs,id_karyawan,tanggal
0,1,9,1171,218,She's good.,None,None
1,2,9,1171,219,He's good.,None,None
2,3,9,1171,147,He's good.,None,None
3,4,20,1501,260,Jojo didn't do the task before the zoom.,None,None
4,5,20,1501,160,Vian didn't do the task before the zoom,None,None



🔍 VALIDASI FK catatan_siswa

  FK ke jadwal:
    Total nilai unik: 155
    Invalid: 0
    ✅ SEMUA VALID!

  FK ke jadwal_detail:
    Total nilai unik: 759
    Invalid: 0
    ✅ SEMUA VALID!

  FK ke siswa:
    Total nilai unik: 375
    Invalid: 0
    ✅ SEMUA VALID!

💾 Menyimpan ke fase_5_afrida.pkl
✅ fase_5_afrida.pkl ditemukan, melanjutkan...
✅ catatan_siswa disimpan ke fase_5_afrida.pkl
   Shape: (1535, 7)
   Total data: 1535 baris
   Mapping id_cs: 1535 entri

✅ Proses catatan_siswa selesai!


In [7]:
display(df_catatan_siswa.head())

,id_cs,id_jadwal,id_jadwal_detail,id_siswa,catatan_cs,id_karyawan,tanggal
0,1,9,1171,218,She's good.,None,None
1,2,9,1171,219,He's good.,None,None
2,3,9,1171,147,He's good.,None,None
3,4,20,1501,260,Jojo didn't do the task before the zoom.,None,None
4,5,20,1501,160,Vian didn't do the task before the zoom,None,None


In [8]:
# Verifikasi isi fase_5_afrida.pkl
with open('fase_5_afrida.pkl', 'rb') as f:
    data = pickle.load(f)

print("\n📋 Isi fase_5_afrida.pkl:")
print(f"  Key: {list(data.keys())}")
print(f"  catatan_siswa shape: {data['catatan_siswa'].shape}")
print(f"  mapping_id_catatan_siswa: {len(data['mapping_id_catatan_siswa'])} entri")

print("\nSample catatan_siswa:")
print(data['catatan_siswa'][['id_cs', 'id_jadwal', 'id_jadwal_detail', 'id_siswa', 'id_karyawan', 'tanggal']].head())


📋 Isi fase_5_afrida.pkl:
  Key: ['presensi_siswa', 'catatan_siswa', 'followup_cs', 'mapping_id_presensi_siswa', 'mapping_id_catatan_siswa']
  catatan_siswa shape: (1535, 7)
  mapping_id_catatan_siswa: 1535 entri

Sample catatan_siswa:
   id_cs  id_jadwal  id_jadwal_detail  id_siswa id_karyawan tanggal
0      1          9              1171       218        None    None
1      2          9              1171       219        None    None
2      3          9              1171       147        None    None
3      4         20              1501       260        None    None
4      5         20              1501       160        None    None


In [9]:
print("\nStruktur catatan_siswa (db_future):")
pd.read_sql("DESCRIBE catatan_siswa", db_future)


Struktur catatan_siswa (db_future):


,Field,Type,Null,Key,Default,Extra
0,id_cs,bigint(20) unsigned,NO,PRI,None,auto_increment
1,id_jadwal,bigint(20) unsigned,YES,MUL,None,
2,id_jadwal_detail,bigint(20) unsigned,YES,MUL,None,
3,id_siswa,bigint(20) unsigned,YES,MUL,None,
4,id_karyawan,bigint(20) unsigned,YES,,None,
5,tanggal,date,YES,,None,
6,catatan_cs,text,NO,,None,


In [10]:
# Cek info dataframe
print("=== Info DataFrame ===")
df_catatan_siswa.info()

print("\n=== Cek NULL per kolom ===")
print(df_catatan_siswa[['id_cs', 'id_jadwal_detail', 'catatan_cs']].isnull().sum())

print("\n=== Tipe data ===")
print(df_catatan_siswa[['id_cs', 'id_jadwal_detail', 'catatan_cs']].dtypes)

print("\n=== Contoh nilai unik id_cs (10 sampel) ===")
print(df_catatan_siswa['id_cs'].head(10))

print("\n=== Apakah id_cs bisa dikonversi ke numerik? ===")
# Cek apakah semua id_cs terdiri dari 'C' diikuti angka?
import re
pattern = r'^C\d+$'
mask = df_catatan_siswa['id_cs'].astype(str).str.match(pattern)
print(f"Jumlah yang sesuai format C<angka>: {mask.sum()} dari {len(df_catatan_siswa)}")
if not mask.all():
    print("Contoh yang tidak sesuai:")
    print(df_catatan_siswa.loc[~mask, 'id_cs'].head())

print("\n=== Cek anomali catatan_cs (misal terlalu panjang atau karakter aneh) ===")
print(f"Panjang maksimal catatan_cs: {df_catatan_siswa['catatan_cs'].str.len().max()}")
print(f"Ada null? {df_catatan_siswa['catatan_cs'].isnull().any()}")

=== Info DataFrame ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1535 entries, 0 to 1534
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id_cs             1535 non-null   int64 
 1   id_jadwal         1535 non-null   int64 
 2   id_jadwal_detail  1535 non-null   int64 
 3   id_siswa          1535 non-null   int64 
 4   catatan_cs        1535 non-null   object
 5   id_karyawan       0 non-null      object
 6   tanggal           0 non-null      object
dtypes: int64(4), object(3)
memory usage: 84.1+ KB

=== Cek NULL per kolom ===
id_cs               0
id_jadwal_detail    0
catatan_cs          0
dtype: int64

=== Tipe data ===
id_cs                int64
id_jadwal_detail     int64
catatan_cs          object
dtype: object

=== Contoh nilai unik id_cs (10 sampel) ===
0     1
1     2
2     3
3     4
4     5
5     6
6     7
7     8
8     9
9    10
Name: id_cs, dtype: int64

=== Apakah id_cs bisa dikonversi 

In [11]:
import pandas as pd
import numpy as np
import pickle

# =========================================================
# 1. LOAD MAPPING YANG DIBUTUHKAN
# =========================================================
print("="*70)
print("📂 Memuat mapping untuk followup_cs")
print("="*70)

# Load mapping_id_catatan_siswa dari fase_5_afrida.pkl
print("\n📂 Memuat fase_5_afrida.pkl...")
try:
    with open('fase_5_afrida.pkl', 'rb') as f:
        data_fase5 = pickle.load(f)
    print("✅ fase_5_afrida.pkl ditemukan")
    
    mapping_id_cs = data_fase5.get('mapping_id_catatan_siswa', {})
    print(f"✅ Mapping id_cs: {len(mapping_id_cs)} entri")
except FileNotFoundError:
    print("❌ fase_5_afrida.pkl tidak ditemukan!")
    print("   Jalankan proses catatan_siswa terlebih dahulu.")
    exit()

# =========================================================
# 2. PROSES DATA followup_cs
# =========================================================
print("\n" + "="*70)
print("⚡ Proses followup_cs (catatan_siswa_follow_up)")
print("="*70)

# 2a. Ambil data dari DB lama
query = "SELECT * FROM catatan_siswa_follow_up"
cursor_old.execute(query)
rows = cursor_old.fetchall()
df_fu = pd.DataFrame(rows)
print(f"Data mentah: {len(df_fu)} baris")
print(f"Kolom asli: {list(df_fu.columns)}")

# =========================================================
# 3. RENAME KOLOM SESUAI TARGET
# =========================================================
rename_map = {
    'idcatatan_siswa': 'id_cs_old',
    'tanggal': 'tanggal_followup',
    'idusers': 'id_user',
    'kesimpulan': 'kesimpulan_followup_cs',
    'status_follow': 'status_followup'
}
# idcs_follow_up tidak di-rename karena akan dihapus (auto increment)
df_fu.rename(columns=rename_map, inplace=True)

# Pilih kolom yang diperlukan (tanpa PK lama)
target_cols = ['id_cs_old', 'tanggal_followup', 'id_user', 'kesimpulan_followup_cs', 'status_followup']
df_fu = df_fu[[col for col in target_cols if col in df_fu.columns]]
print(f"Setelah mapping kolom: {list(df_fu.columns)}")

# =========================================================
# 4. MAPPING id_cs (old -> new)
# =========================================================
print("\n📌 Mapping id_cs...")
df_fu['id_cs'] = df_fu['id_cs_old'].astype(str).map(mapping_id_cs)

before = len(df_fu)
df_fu.dropna(subset=['id_cs'], inplace=True)
print(f"  Baris orphan (id_cs tidak valid) dibuang: {before - len(df_fu)}")
df_fu['id_cs'] = df_fu['id_cs'].astype('int64')

# Hapus kolom old id
df_fu.drop(columns=['id_cs_old'], inplace=True)

# =========================================================
# 5. KONVERSI TIPE DATA
# =========================================================
print("\n📌 Konversi tipe data...")

# Konversi tanggal
df_fu['tanggal_followup'] = pd.to_datetime(df_fu['tanggal_followup'], errors='coerce')
failed_conv = df_fu['tanggal_followup'].isnull().sum()
if failed_conv > 0:
    print(f"  ⚠️ {failed_conv} baris gagal konversi ke datetime (dijadi NaT)")

# id_user tetap string (akan di-mapping nanti di fase users)
df_fu['id_user'] = df_fu['id_user'].astype(str).replace('nan', np.nan)

# status_followup: pastikan enum valid
expected_enum = ['NEED FURTHER OBSERVATION', 'CASE CLOSED']
df_fu['status_followup'] = df_fu['status_followup'].fillna('NEED FURTHER OBSERVATION')
# Validasi nilai enum
invalid_enum = set(df_fu['status_followup'].unique()) - set(expected_enum)
if invalid_enum:
    print(f"  ⚠️ Nilai enum tidak valid: {invalid_enum}")

# kesimpulan: fillna
df_fu['kesimpulan_followup_cs'] = df_fu['kesimpulan_followup_cs'].fillna('').astype(str)

# =========================================================
# 6. TAMBAHKAN ID INTERNAL & MAPPING
# =========================================================
print("\n📌 Generate id_followup_cs baru...")

# Urutkan berdasarkan data yang ada
df_fu.sort_values('tanggal_followup', ascending=True, inplace=True)
df_fu.reset_index(drop=True, inplace=True)

# Tambahkan id (auto increment)
df_fu.insert(0, 'id', range(1, len(df_fu) + 1))

# Simpan mapping old_id -> new_id (jika diperlukan)
# mapping_id_followup_cs = dict(zip(old_ids, df_fu['id']))

# =========================================================
# 7. FINALISASI
# =========================================================
print("\n📌 Finalisasi...")

# Susun ulang kolom
kolom_final = ['id', 'id_cs', 'tanggal_followup', 'id_user', 'kesimpulan_followup_cs', 'status_followup']
df_fu = df_fu[kolom_final]

# =========================================================
# 8. HASIL AKHIR
# =========================================================
df_followup_cs = df_fu.copy()

print(f"\n✅ df_followup_cs siap. Shape: {df_followup_cs.shape}")
print(f"   Kolom: {df_followup_cs.columns.tolist()}")
print(f"   Tipe id_cs: {df_followup_cs['id_cs'].dtype}")

print("\n📋 Sample data followup_cs:")
display(df_followup_cs.head())

# =========================================================
# 9. VALIDASI FK
# =========================================================
print("\n" + "="*70)
print("🔍 VALIDASI FK followup_cs")
print("="*70)

# Validasi ke catatan_siswa (id_cs)
fk_cs_values = set(df_followup_cs['id_cs'])
valid_cs_keys = set(mapping_id_cs.values())
invalid_cs = fk_cs_values - valid_cs_keys
print(f"\n  FK ke catatan_siswa (id_cs):")
print(f"    Total nilai unik: {len(fk_cs_values)}")
print(f"    Invalid: {len(invalid_cs)}")
if len(invalid_cs) == 0:
    print("    ✅ SEMUA VALID!")
else:
    print(f"    ❌ Invalid: {list(invalid_cs)[:5]}")

# =========================================================
# 10. SIMPAN KE fase_5_afrida.pkl
# =========================================================
print("\n" + "="*70)
print("💾 Menyimpan ke fase_5_afrida.pkl")
print("="*70)

# Load file fase_5_afrida.pkl jika ada
try:
    with open('fase_5_afrida.pkl', 'rb') as f:
        data_fase5 = pickle.load(f)
    print("✅ fase_5_afrida.pkl ditemukan, melanjutkan...")
except FileNotFoundError:
    data_fase5 = {}
    print("📂 fase_5_afrida.pkl belum ada, membuat baru...")

# Tambahkan data followup_cs
data_fase5['followup_cs'] = df_followup_cs

# Simpan ke file
with open('fase_5_afrida.pkl', 'wb') as f:
    pickle.dump(data_fase5, f)

print(f"✅ followup_cs disimpan ke fase_5_afrida.pkl")
print(f"   Shape: {df_followup_cs.shape}")
print(f"   Total data: {len(df_followup_cs)} baris")

print("\n✅ Proses followup_cs selesai!")

📂 Memuat mapping untuk followup_cs

📂 Memuat fase_5_afrida.pkl...
✅ fase_5_afrida.pkl ditemukan
✅ Mapping id_cs: 1535 entri

⚡ Proses followup_cs (catatan_siswa_follow_up)
Data mentah: 22 baris
Kolom asli: ['idcs_follow_up', 'idcatatan_siswa', 'tanggal', 'idusers', 'kesimpulan', 'status_follow']
Setelah mapping kolom: ['id_cs_old', 'tanggal_followup', 'id_user', 'kesimpulan_followup_cs', 'status_followup']

📌 Mapping id_cs...
  Baris orphan (id_cs tidak valid) dibuang: 0

📌 Konversi tipe data...

📌 Generate id_followup_cs baru...

📌 Finalisasi...

✅ df_followup_cs siap. Shape: (22, 6)
   Kolom: ['id', 'id_cs', 'tanggal_followup', 'id_user', 'kesimpulan_followup_cs', 'status_followup']
   Tipe id_cs: int64

📋 Sample data followup_cs:


,id,id_cs,tanggal_followup,id_user,kesimpulan_followup_cs,status_followup
0,1,12,2023-07-14,U00026,"Okay, bantu FU - Qorin",NEED FURTHER OBSERVATION
1,2,56,2023-07-14,U00011,"done keluarkan LV dan WAG yah, Sarah akan kemb...",NEED FURTHER OBSERVATION
2,3,59,2023-07-14,U00011,"Rehan blm bayar SPP, sudah di japri Daniar blm...",NEED FURTHER OBSERVATION
3,4,63,2023-07-20,U00011,"sudah masuk, dan mama sudah bersedia ditagih S...",CASE CLOSED
4,5,132,2023-07-28,U00011,"(CS28)\r\nMiss Daniar , ini jika nanti Miss Ri...",NEED FURTHER OBSERVATION



🔍 VALIDASI FK followup_cs

  FK ke catatan_siswa (id_cs):
    Total nilai unik: 22
    Invalid: 0
    ✅ SEMUA VALID!

💾 Menyimpan ke fase_5_afrida.pkl
✅ fase_5_afrida.pkl ditemukan, melanjutkan...
✅ followup_cs disimpan ke fase_5_afrida.pkl
   Shape: (22, 6)
   Total data: 22 baris

✅ Proses followup_cs selesai!


In [12]:
display(df_followup_cs.head())

,id,id_cs,tanggal_followup,id_user,kesimpulan_followup_cs,status_followup
0,1,12,2023-07-14,U00026,"Okay, bantu FU - Qorin",NEED FURTHER OBSERVATION
1,2,56,2023-07-14,U00011,"done keluarkan LV dan WAG yah, Sarah akan kemb...",NEED FURTHER OBSERVATION
2,3,59,2023-07-14,U00011,"Rehan blm bayar SPP, sudah di japri Daniar blm...",NEED FURTHER OBSERVATION
3,4,63,2023-07-20,U00011,"sudah masuk, dan mama sudah bersedia ditagih S...",CASE CLOSED
4,5,132,2023-07-28,U00011,"(CS28)\r\nMiss Daniar , ini jika nanti Miss Ri...",NEED FURTHER OBSERVATION


In [13]:
# Verifikasi isi fase_5_afrida.pkl
with open('fase_5_afrida.pkl', 'rb') as f:
    data = pickle.load(f)

print("\n📋 Isi fase_5_afrida.pkl:")
print(f"  Key: {list(data.keys())}")
print(f"  followup_cs shape: {data['followup_cs'].shape}")

print("\nSample followup_cs:")
print(data['followup_cs'].head())


📋 Isi fase_5_afrida.pkl:
  Key: ['presensi_siswa', 'catatan_siswa', 'followup_cs', 'mapping_id_presensi_siswa', 'mapping_id_catatan_siswa']
  followup_cs shape: (22, 6)

Sample followup_cs:
   id  id_cs tanggal_followup id_user  \
0   1     12       2023-07-14  U00026   
1   2     56       2023-07-14  U00011   
2   3     59       2023-07-14  U00011   
3   4     63       2023-07-20  U00011   
4   5    132       2023-07-28  U00011   

                              kesimpulan_followup_cs           status_followup  
0                             Okay, bantu FU - Qorin  NEED FURTHER OBSERVATION  
1  done keluarkan LV dan WAG yah, Sarah akan kemb...  NEED FURTHER OBSERVATION  
2  Rehan blm bayar SPP, sudah di japri Daniar blm...  NEED FURTHER OBSERVATION  
3  sudah masuk, dan mama sudah bersedia ditagih S...               CASE CLOSED  
4  (CS28)\r\nMiss Daniar , ini jika nanti Miss Ri...  NEED FURTHER OBSERVATION  


In [14]:
print("\nStruktur FOLLOWUP_CS (db_future):")
pd.read_sql("DESCRIBE followup_cs", db_future)


Struktur FOLLOWUP_CS (db_future):


,Field,Type,Null,Key,Default,Extra
0,id_followup_cs,bigint(20) unsigned,NO,PRI,None,auto_increment
1,id_cs,bigint(20) unsigned,YES,MUL,None,
2,tanggal_followup,timestamp,NO,,current_timestamp(),
3,id_user,varchar(20),YES,MUL,None,
4,kesimpulan_followup_cs,text,NO,,None,
5,status_followup,"enum('NEED FURTHER OBSERVATION','CASE CLOSED')",NO,,NEED FURTHER OBSERVATION,


In [16]:
import pickle
import pandas as pd

# =========================================================
# VERSI RINGKAS: SAVE fase_5_afrida.pkl
# =========================================================
print("🧹 Membersihkan & menyimpan fase_5_afrida.pkl...")

# Daftar dataframe dan key
data_fase5 = {}
df_mapping = {
    'df_presensi_siswa': 'presensi_siswa',
    'df_catatan_siswa': 'catatan_siswa',
    'df_followup_cs': 'followup_cs'
}

# Proses setiap dataframe
for df_var, key in df_mapping.items():
    if df_var in locals():
        df = locals()[df_var]
        # Hapus kolom 'id' jika ada
        if 'id' in df.columns:
            df = df.drop(columns=['id'])
            print(f"  ✅ {key}: kolom 'id' dihapus")
        data_fase5[key] = df
    else:
        print(f"  ⚠️ {df_var} tidak ditemukan, dilewati")

# Simpan ke file
with open('fase_5_afrida.pkl', 'wb') as f:
    pickle.dump(data_fase5, f)

print(f"✅ fase_5_afrida.pkl selesai! Keys: {list(data_fase5.keys())}")

🧹 Membersihkan & menyimpan fase_5_afrida.pkl...
  ✅ presensi_siswa: kolom 'id' dihapus
  ✅ followup_cs: kolom 'id' dihapus
✅ fase_5_afrida.pkl selesai! Keys: ['presensi_siswa', 'catatan_siswa', 'followup_cs']


## 3. Transform Data (jika diperlukan)

## 4. Insert ke DB Baru

## 5. Verifikasi Data

## 6. Return Hasil Migrasi untuk migrate_db.py

## 7. Close Connection